# Appendix: Chronos-2 — data gaps & the effect of covariates

Two questions the core notebook glosses over:

1. **What happens when the target history has gaps?** Real observation records
   have missing days. Chronos-2 is trained to forecast from incomplete context and
   ingests NaNs directly, but the effect on skill is worth seeing. We take a
   clean series, punch synthetic gaps into it, and re-forecast.
2. **How much do covariates change the forecast?** We compare a target-only
   forecast against one conditioned on a future-known covariate.

This is a diagnostic notebook — it deliberately runs on synthetic data so it
stays fast in a live session and needs no downloads beyond the model.


In [ ]:
# CPU torch first so the chronos install doesn't pull the large CUDA torch build;
# then numpy pinned so later installs don't leave a half-upgraded numpy. If a numpy
# ImportError appears on Colab, restart & re-run.
!pip install -q "torch<2.10" --index-url https://download.pytorch.org/whl/cpu
!pip install -q "numpy>=1.26,<2.1" "chronos-forecasting[extras]>=2.2" pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# A synthetic but realistic daily series: seasonal cycle + AR(1) noise.
n = 365 * 4
rng = np.random.default_rng(0)
t = np.arange(n)
seasonal = 20 + 15 * np.sin(2 * np.pi * t / 365)
noise = np.zeros(n)
for i in range(1, n):
    noise[i] = 0.8 * noise[i - 1] + rng.normal(0, 3)
series = np.clip(seasonal + noise, 1, None)
idx = pd.date_range("2022-01-01", periods=n, freq="D")
clean = pd.Series(series, index=idx, name="value")

HORIZON = 15
train = clean.iloc[:-HORIZON]
truth = clean.iloc[-HORIZON:]


## Load Chronos-2

In [ ]:
from chronos import Chronos2Pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
chronos = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device)
print("Chronos-2 loaded on", device)


## Forecast helper (target-only)

To isolate the gap effect we forecast the univariate target with no covariates.
Chronos-2 accepts a bare context series with NaNs.


In [ ]:
def chronos_forecast(context_series):
    # predict_df wants a long frame with an id, a timestamp, and the target column.
    context = (context_series.rename("value").rename_axis("timestamp")
               .reset_index().assign(item="series"))
    pred = chronos.predict_df(
        context, prediction_length=HORIZON, quantile_levels=[0.5],
        id_column="item", timestamp_column="timestamp", target="value",
    )
    return pred.sort_values("timestamp")["0.5"].to_numpy()  # median quantile


## Punch synthetic gaps

We drop random days from the *recent* history (last year) at increasing rates and
re-forecast. Chronos-2 accepts the NaNs directly — no imputation needed.


In [ ]:
def with_gaps(series, frac, seed):
    r = np.random.default_rng(seed)
    s = series.copy()
    recent = s.index[-365:]
    drop = r.choice(recent, size=int(frac * len(recent)), replace=False)
    s.loc[drop] = np.nan
    return s

rmse = lambda a, b: float(np.sqrt(np.nanmean((np.asarray(a) - np.asarray(b)) ** 2)))

records = []
for frac in [0.0, 0.1, 0.3, 0.5, 0.7]:
    gapped = with_gaps(train, frac, seed=1)
    pred = chronos_forecast(gapped)                # native NaN handling
    records.append({"gap_frac": frac, "Chronos-2 RMSE": rmse(pred, truth.values)})
skill = pd.DataFrame(records).set_index("gap_frac")
skill


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
skill.plot(marker="o", ax=ax, legend=False)
ax.set_xlabel("Fraction of last-year history missing")
ax.set_ylabel(f"{HORIZON}-day forecast RMSE")
ax.set_title("Chronos-2 forecast skill vs. gaps in the target history")


## Takeaways

- Chronos-2 ingests NaNs natively and degrades gracefully as gaps grow, rather than
  failing — no imputation step required.
- This is why the core notebook keeps **target** gaps and only requires the
  **covariates** to be present: the model handles a patchy observation record for
  you.
- For a very gappy or short record, expect wider uncertainty; consider a longer
  history window (`HISTORY_DAYS` in the core notebook) to give the model more
  seasonal context.
